# Minimal RAG Pipeline

This notebook loads documents, chunks them, stores embeddings in Pinecone, retrieves similar chunks for a query, and generates an answer with an LLM.


## Imports

This cell loads the shared helpers from `rag_pipeline.py` and the small utilities needed in the notebook.


In [ ]:
from pathlib import Path

from rag_pipeline import (
    DEFAULT_CHAT_MODEL,
    DEFAULT_EMBEDDING_MODEL,
    PINECONE_INDEX_NAME,
    answer_query,
    build_chunks,
    embed_texts,
    index_documents,
    retrieve_context,
)


## Environment and Config

This cell defines the document directory, namespace, and query used throughout the notebook.


In [ ]:
documents_dir = Path("./docs")
namespace = "default"
query = "What does the paper say about hybrid rag?"

print(f"Chat model: {DEFAULT_CHAT_MODEL}")
print(f"Embedding model: {DEFAULT_EMBEDDING_MODEL}")
print(f"Pinecone index: {PINECONE_INDEX_NAME}")
print(f"Documents directory: {documents_dir.resolve()}")


## Document Loading and Chunking

This section reads text from each supported file type and splits it into overlapping chunks for embedding.


In [ ]:
chunks = build_chunks(documents_dir)
print(f"Loaded {len(chunks)} chunks.")
if chunks:
    print("First chunk source:", chunks[0].source)
    print("First chunk preview:", chunks[0].text[:300])


## Pinecone and Embeddings

This section creates the embedding model and converts text into vectors.


In [ ]:
if chunks:
    sample_embedding = embed_texts([chunks[0].text])[0]
    print(f"Sample embedding dimension: {len(sample_embedding)}")
else:
    print("No supported documents were found in ./docs.")


## Indexing and Retrieval

This section stores chunks in Pinecone and retrieves the nearest matches for a query.


In [ ]:
indexed = index_documents(documents_dir, namespace=namespace)
print(f"Indexed {indexed} chunks.")

contexts = retrieve_context(query, top_k=5, namespace=namespace)
for item in contexts:
    print(f"[{item['score']:.4f}] {item['source']}#{item['chunk_index']}")
    print(item["text"])
    print()


## Example Usage

This final cell shows the end-to-end question answering step using the retrieved context.


In [ ]:
answer = answer_query(query, top_k=5, namespace=namespace, contexts=contexts)
print(answer)
